In [1]:
import os
import re
import json
import math
import random
from copy import deepcopy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import scipy.io as sio
from scipy.stats import spearmanr

# ==============================================================================
# Repro + device
# ==============================================================================
SEED = 420

def set_seed(seed: int = 420):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[info] SEED={SEED} | DEVICE={DEVICE}")

#------------------------------------------------------------------------------
# Paths + configuration
#------------------------------------------------------------------------------

BASE_DATA_DIR = Path("data")

# Wikipedia corpus directory with JSONL files (Wikimedia Enterprise format)
WIKI_DIR = BASE_DATA_DIR / "enwiki_namespace_0"     # 74GB JSONL corpus

# THINGS similarity bundle
THINGS_DIR = Path("things_similarity")
THINGS_WORDS_PATH = THINGS_DIR / "variables" / "unique_id.txt"
BEHAVIORAL_SIM_PATH = THINGS_DIR / "data" / "spose_similarity.mat"

# SimLex-999 (CSV or TSV; adjust if yours differs)
SIMLEX_PATH = Path("Simlex-999") / "SimLex-999.txt"
  # common filename; update if needed

# Output
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

#------------------------------------------------------------------------------
# Hyperparameters (reasonable defaults; tune freely)
#------------------------------------------------------------------------------

# Vocab / tokenization
MIN_COUNT = 30
MAX_VOCAB = None          # or int like 200_000

# Word2Vec
EMBEDDING_DIM = 300 #Google News 300
WINDOW_SIZE = 5
NEG_SAMPLES = 5
SUBSAMPLE_T = 1e-5    # set None to disable subsampling

# Training schedule
EPOCHS = 2
BATCH_SIZE = 8192
LR = 2e-3
BATCHES_PER_EPOCH = 10000   # caps training per epoch (streaming corpus)

# RSR (THINGS similarity alignment)
RSR_WEIGHT = 0.01
RSR_EVERY_N_BATCHES = 5
RSR_PAIRS_PER_STEP = 10000
SOFT_RANK_STRENGTH = 2.0

# Optional: THINGS-focused W2V (leave 0.0 if you only want "add one THINGS loss" via RSR)
THINGS_W2V_WEIGHT = 0.0

# Corpus loading limits (for debugging)
MAX_JSON_FILES = None         # e.g. 2 for quick test
MAX_ARTICLES = None           # e.g. 5000 for quick test

print("[config]")
print(f"  WIKI_DIR={WIKI_DIR}")
print(f"  MIN_COUNT={MIN_COUNT} | EMBEDDING_DIM={EMBEDDING_DIM} | WINDOW={WINDOW_SIZE} | NEG={NEG_SAMPLES}")
print(f"  EPOCHS={EPOCHS} | BATCH_SIZE={BATCH_SIZE} | LR={LR} | BATCHES_PER_EPOCH={BATCHES_PER_EPOCH}")
print(f"  RSR_WEIGHT={RSR_WEIGHT} | RSR_EVERY={RSR_EVERY_N_BATCHES} | RSR_PAIRS={RSR_PAIRS_PER_STEP}")
print(f"  THINGS_W2V_WEIGHT={THINGS_W2V_WEIGHT}")


[info] SEED=420 | DEVICE=cuda
[config]
  WIKI_DIR=data\enwiki_namespace_0
  MIN_COUNT=30 | EMBEDDING_DIM=300 | WINDOW=5 | NEG=5
  EPOCHS=2 | BATCH_SIZE=8192 | LR=0.002 | BATCHES_PER_EPOCH=10000
  RSR_WEIGHT=0.01 | RSR_EVERY=5 | RSR_PAIRS=10000
  THINGS_W2V_WEIGHT=0.0


In [ ]:
#------------------------------------------------------------------------------
# Tokenization + streaming Wikipedia sentence iterator
#------------------------------------------------------------------------------

_token_re = re.compile(r"[^a-zA-Z\s]+")

def simple_tokenize(text: str):
    text = text.lower()
    text = _token_re.sub(" ", text)
    return text.split()

def iter_wiki_sentences_jsonl(
    wiki_dir: Path,
    max_files=None,
    max_articles=None,
):
    """
    Stream tokenized sentences from JSONL files (Wikimedia Enterprise format).
    Each line is a JSON object with nested sections containing paragraphs.
    
    This is streaming to avoid holding the whole corpus in RAM.
    """
    jsonl_files = sorted(wiki_dir.glob("*.jsonl"))
    if max_files is not None:
        jsonl_files = jsonl_files[:max_files]

    article_count = 0
    for jf in tqdm(jsonl_files, desc="Processing JSONL files", leave=False):
        with jf.open("r", encoding="utf-8") as f:
            for line in f:
                if max_articles is not None and article_count >= max_articles:
                    return
                
                try:
                    art = json.loads(line)
                except:
                    continue
                
                article_count += 1
                
                # Extract text from nested sections (Wikimedia Enterprise format)
                for section in art.get("sections", []):
                    for part in section.get("has_parts", []):
                        if part.get("type") == "paragraph":
                            text = part.get("value", "")
                            if text:
                                # cheap sentence split; good enough for W2V
                                for sent in text.split(". "):
                                    toks = simple_tokenize(sent)
                                    if len(toks) >= 2:
                                        yield toks

def sentence_stream_factory():
    # Factory that returns a *fresh* generator each time (important for multiple passes / epochs)
    return iter_wiki_sentences_jsonl(
        WIKI_DIR,
        max_files=MAX_JSON_FILES,
        max_articles=MAX_ARTICLES,
    )

In [3]:
#------------------------------------------------------------------------------
# Build vocab (1st pass over the corpus)
#------------------------------------------------------------------------------

def build_vocab_from_stream(stream, min_count=5, max_vocab=None):
    counts = Counter()
    for toks in tqdm(stream, desc="Counting vocab (stream pass 1)"):
        counts.update(toks)

    # Filter + sort
    items = [(w, c) for w, c in counts.items() if c >= min_count]
    items.sort(key=lambda x: x[1], reverse=True)
    if max_vocab is not None:
        items = items[:max_vocab]

    # Reserve 0 for <UNK> to make filtering easier
    vocab = ["<UNK>"] + [w for w, _ in items]
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}

    # counts aligned to vocab indices
    idx_counts = np.zeros(len(vocab), dtype=np.int64)
    idx_counts[0] = 1  # arbitrary for <UNK>
    for w, c in items:
        idx_counts[word2idx[w]] = c

    return word2idx, idx2word, idx_counts

print("\n" + "=" * 70)
print("STEP 1: Build vocabulary (streaming over Wikipedia)")
print("=" * 70)

word2idx, idx2word, idx_counts = build_vocab_from_stream(
    sentence_stream_factory(),
    min_count=MIN_COUNT,
    max_vocab=MAX_VOCAB,
)

VOCAB_SIZE = len(word2idx)
print(f"[info] vocab_size={VOCAB_SIZE:,} (min_count={MIN_COUNT})")


STEP 1: Build vocabulary (streaming over Wikipedia)


Counting vocab (stream pass 1): 107735489it [14:38, 122598.54it/s]


[info] vocab_size=507,578 (min_count=30)


In [4]:
#------------------------------------------------------------------------------
# Subsampling + negative sampling distributions
#------------------------------------------------------------------------------

def make_unigram_dist(idx_counts: np.ndarray, power: float = 0.75) -> torch.Tensor:
    """
    Negative sampling distribution ~ count^0.75
    Returns a normalized torch tensor on CPU (use torch.multinomial for sampling).
    """
    freqs = idx_counts.astype(np.float64)
    freqs[0] = 0.0  # don't sample <UNK> as negative
    p = np.power(freqs, power)
    p = p / (p.sum() + 1e-12)
    return torch.tensor(p, dtype=torch.float32)

NEG_DIST = make_unigram_dist(idx_counts)

def make_subsampling_keep_probs(idx_counts: np.ndarray, t: float = 1e-5) -> np.ndarray:
    """
    Mikolov subsampling: keep_prob = min(1, sqrt(t/f) + t/f)
    where f is word frequency.
    """
    freqs = idx_counts / idx_counts.sum()
    keep = np.ones_like(freqs, dtype=np.float64)
    # avoid div0 and ignore <UNK>
    mask = freqs > 0
    keep[mask] = np.minimum(1.0, (np.sqrt(t / freqs[mask]) + (t / freqs[mask])))
    keep[0] = 0.0
    return keep

KEEP_PROB = None
if SUBSAMPLE_T is not None:
    KEEP_PROB = make_subsampling_keep_probs(idx_counts, t=SUBSAMPLE_T)

def tokens_to_indices(tokens):
    """
    Map tokens -> indices, filter OOV to <UNK> (then we drop <UNK>).
    Apply optional subsampling.
    """
    idxs = []
    for w in tokens:
        i = word2idx.get(w, 0)
        if i == 0:
            continue
        if KEEP_PROB is not None:
            if random.random() > KEEP_PROB[i]:
                continue
        idxs.append(i)
    return idxs

In [5]:
#------------------------------------------------------------------------------
# Generate skip-gram pairs (streaming) + batch builder
#------------------------------------------------------------------------------

def iter_skipgram_pairs(sentence_stream, window_size=5):
    """
    Yields (target_idx, context_idx) pairs for skip-gram training.
    Streaming; does not store corpus.
    """
    for toks in sentence_stream:
        idxs = tokens_to_indices(toks)
        if len(idxs) < 2:
            continue
        for center_pos, target in enumerate(idxs):
            left = max(0, center_pos - window_size)
            right = min(len(idxs), center_pos + window_size + 1)
            for ctx_pos in range(left, right):
                if ctx_pos == center_pos:
                    continue
                yield target, idxs[ctx_pos]

def batch_pairs(pair_iter, batch_size):
    """
    Yield batches of (targets, pos_contexts) as torch tensors.
    """
    targets = []
    contexts = []
    for t, c in pair_iter:
        targets.append(t)
        contexts.append(c)
        if len(targets) >= batch_size:
            yield torch.tensor(targets, dtype=torch.long), torch.tensor(contexts, dtype=torch.long)
            targets, contexts = [], []
    # drop remainder for simplicity (fine for streaming)

In [6]:
#------------------------------------------------------------------------------
# Model: Skip-gram with negative sampling (FROM SCRATCH)
#------------------------------------------------------------------------------

class SkipGramWord2Vec(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

        # Init similar to word2vec-ish small uniform
        bound = 0.5 / embedding_dim
        nn.init.uniform_(self.in_embed.weight, -bound, bound)
        nn.init.uniform_(self.out_embed.weight, -bound, bound)

    def forward(self, target_idx, pos_ctx_idx, neg_ctx_idx):
        """
        target_idx: (B,)
        pos_ctx_idx: (B,)
        neg_ctx_idx: (B, K)
        """
        v = self.in_embed(target_idx)                 # (B, D)
        u_pos = self.out_embed(pos_ctx_idx)           # (B, D)
        u_neg = self.out_embed(neg_ctx_idx)           # (B, K, D)

        pos_logits = (v * u_pos).sum(dim=1)           # (B,)
        neg_logits = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)  # (B, K)

        return pos_logits, neg_logits

def w2v_neg_sampling_loss(pos_logits, neg_logits):
    """
    Negative sampling objective:
      log sigma(pos) + sum_k log sigma(-neg_k)
    (we minimize negative of that)
    """
    pos_loss = F.logsigmoid(pos_logits).mean()
    neg_loss = F.logsigmoid(-neg_logits).mean()
    return -(pos_loss + neg_loss)


In [7]:
#------------------------------------------------------------------------------
# RSR: soft rank + soft Spearman (differentiable)
#------------------------------------------------------------------------------

def soft_rank(x: torch.Tensor, regularization_strength: float = 1.0) -> torch.Tensor:
    """
    Differentiable approximation to ranks using pairwise sigmoid comparisons.
    x: (n,)
    returns: (n,)
    """
    x = x.flatten()
    diffs = x.unsqueeze(1) - x.unsqueeze(0)   # (n, n)
    soft_comparisons = torch.sigmoid(regularization_strength * diffs)
    ranks = soft_comparisons.sum(dim=1)
    return ranks

def soft_spearman(pred: torch.Tensor, target: torch.Tensor, regularization_strength: float = 1.0) -> torch.Tensor:
    """
    Differentiable Spearman correlation via soft ranks.
    Returns correlation ~ [-1, 1]
    """
    pr = soft_rank(pred, regularization_strength)
    tr = soft_rank(target, regularization_strength)

    pr = pr - pr.mean()
    tr = tr - tr.mean()

    pr = pr / (pr.norm() + 1e-8)
    tr = tr / (tr.norm() + 1e-8)

    return (pr * tr).sum()

In [8]:
#------------------------------------------------------------------------------
# THINGS loader + RSR pair sampler
#------------------------------------------------------------------------------

def load_things_words(path: Path):
    """
    THINGS unique_id.txt typically includes one concept per line.
    """
    with path.open("r", encoding="utf-8") as f:
        words = [ln.strip() for ln in f if ln.strip()]
    # normalize to match tokenization
    words = [w.lower() for w in words]
    return words

def load_spose_similarity(path: Path):
    """
    Loads spose_similarity.mat and returns similarity matrix.
    Variable name is often 'spose_sim' or similar; we defensively pick the first 2D array.
    """
    mat = sio.loadmat(path)
    # Find first 2D numeric array
    for k, v in mat.items():
        if k.startswith("__"):
            continue
        if isinstance(v, np.ndarray) and v.ndim == 2 and v.shape[0] == v.shape[1]:
            return v
    raise ValueError(f"Could not find square similarity matrix in {path}")

print("\n" + "=" * 70)
print("STEP 2: Load THINGS + behavioral similarity")
print("=" * 70)

things_words = load_things_words(THINGS_WORDS_PATH)
spose_sim = load_spose_similarity(BEHAVIORAL_SIM_PATH)

# Build mapping: THINGS concepts -> vocab indices (only those present in our vocab)
valid_concepts = []
valid_vocab_indices = []
valid_things_indices = []

things_word_to_i = {w: i for i, w in enumerate(things_words)}
for w in things_words:
    vi = word2idx.get(w, 0)
    if vi != 0:
        valid_concepts.append(w)
        valid_vocab_indices.append(vi)
        valid_things_indices.append(things_word_to_i[w])

valid_vocab_indices = np.array(valid_vocab_indices, dtype=np.int64)
valid_things_indices = np.array(valid_things_indices, dtype=np.int64)

print(f"[info] THINGS concepts total={len(things_words):,}")
print(f"[info] THINGS concepts in vocab={len(valid_concepts):,}")

# Extract the submatrix of behavioral similarity for the concepts we can train/eval on
THINGS_SIM_SUB = spose_sim[np.ix_(valid_things_indices, valid_things_indices)].astype(np.float32)

# Precompute upper-tri indices for uniform random pair sampling
# (exclude diagonal)
N_TH = THINGS_SIM_SUB.shape[0]
tri_u = np.triu_indices(N_TH, k=1)
ALL_PAIR_COUNT = len(tri_u[0])
print(f"[info] Available THINGS pairs (upper-tri)={ALL_PAIR_COUNT:,}")

def sample_things_pairs(num_pairs: int):
    """
    Returns:
      vocab_i: (P,)
      vocab_j: (P,)
      target_sim: (P,)
    """
    idx = np.random.randint(0, ALL_PAIR_COUNT, size=num_pairs)
    ai = tri_u[0][idx]
    aj = tri_u[1][idx]

    vocab_i = valid_vocab_indices[ai]
    vocab_j = valid_vocab_indices[aj]
    target_sim = THINGS_SIM_SUB[ai, aj]

    return (
        torch.tensor(vocab_i, dtype=torch.long, device=DEVICE),
        torch.tensor(vocab_j, dtype=torch.long, device=DEVICE),
        torch.tensor(target_sim, dtype=torch.float32, device=DEVICE),
    )


STEP 2: Load THINGS + behavioral similarity
[info] THINGS concepts total=1,854
[info] THINGS concepts in vocab=1,486
[info] Available THINGS pairs (upper-tri)=1,103,355


In [9]:
#------------------------------------------------------------------------------
# Training helpers
#------------------------------------------------------------------------------

@torch.no_grad()
def cosine_sim_from_in_embeddings(model: SkipGramWord2Vec, idx_a: torch.Tensor, idx_b: torch.Tensor):
    """
    cosine similarity between in-embeddings (common choice for W2V)
    idx_a/idx_b: (P,)
    returns: (P,)
    """
    va = model.in_embed(idx_a)
    vb = model.in_embed(idx_b)
    va = va / (va.norm(dim=1, keepdim=True) + 1e-8)
    vb = vb / (vb.norm(dim=1, keepdim=True) + 1e-8)
    return (va * vb).sum(dim=1)

def cosine_sim_from_in_embeddings_grad(model: SkipGramWord2Vec, idx_a: torch.Tensor, idx_b: torch.Tensor):
    """
    same as above, but with gradients enabled (for RSR loss).
    """
    va = model.in_embed(idx_a)
    vb = model.in_embed(idx_b)
    va = va / (va.norm(dim=1, keepdim=True) + 1e-8)
    vb = vb / (vb.norm(dim=1, keepdim=True) + 1e-8)
    return (va * vb).sum(dim=1)

def train_one_epoch_streaming(
    model: SkipGramWord2Vec,
    optimizer: optim.Optimizer,
    sentence_stream_fn,
    batches_per_epoch: int,
    batch_size: int,
    window_size: int,
    neg_samples: int,
    neg_dist: torch.Tensor,
    rsr_weight: float = 0.0,
    rsr_every_n: int = 10,
    rsr_pairs_per_step: int = 5000,
    soft_rank_strength: float = 2.0,
    things_w2v_weight: float = 0.0,
):
    model.train()

    # streaming pair iterator (fresh)
    pair_iter = iter_skipgram_pairs(sentence_stream_fn(), window_size=window_size)
    batch_iter = batch_pairs(pair_iter, batch_size=batch_size)

    total_loss = 0.0
    total_w2v = 0.0
    total_rsr = 0.0
    total_thw2v = 0.0

    pbar = tqdm(range(batches_per_epoch), desc="training", leave=False)
    for b in pbar:
        try:
            tgt, ctx = next(batch_iter)
        except StopIteration:
            # corpus exhausted; restart stream mid-epoch (fine for huge corpora)
            pair_iter = iter_skipgram_pairs(sentence_stream_fn(), window_size=window_size)
            batch_iter = batch_pairs(pair_iter, batch_size=batch_size)
            tgt, ctx = next(batch_iter)

        tgt = tgt.to(DEVICE)
        ctx = ctx.to(DEVICE)

        # negatives: (B, K)
        neg = torch.multinomial(neg_dist, num_samples=tgt.shape[0] * neg_samples, replacement=True)
        neg = neg.view(tgt.shape[0], neg_samples).to(DEVICE)

        pos_logits, neg_logits = model(tgt, ctx, neg)
        loss_w2v = w2v_neg_sampling_loss(pos_logits, neg_logits)

        # Optional THINGS-focused W2V term (kept as a simple hook; default 0.0)
        # Here we just reuse the same w2v loss; in a more elaborate setup you’d bias sampling
        # toward THINGS words/sentences.
        loss_things_w2v = loss_w2v.detach() * 0.0
        if things_w2v_weight > 0.0:
            loss_things_w2v = loss_w2v  # placeholder: you can implement a THINGS-biased sampler later

        # RSR term (every N batches)
        loss_rsr = torch.tensor(0.0, device=DEVICE)
        if rsr_weight > 0.0 and (b % rsr_every_n == 0):
            i_idx, j_idx, target_sim = sample_things_pairs(rsr_pairs_per_step)
            pred_sim = cosine_sim_from_in_embeddings_grad(model, i_idx, j_idx)
            rho = soft_spearman(pred_sim, target_sim, regularization_strength=soft_rank_strength)
            loss_rsr = 1.0 - rho  # maximize rho -> minimize (1-rho)

        loss = loss_w2v + (things_w2v_weight * loss_things_w2v) + (rsr_weight * loss_rsr)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        total_w2v += float(loss_w2v.item())
        total_rsr += float(loss_rsr.item()) if rsr_weight > 0.0 else 0.0
        total_thw2v += float(loss_things_w2v.item()) if things_w2v_weight > 0.0 else 0.0

        if (b + 1) % 200 == 0:
            pbar.set_postfix({
                "loss": total_loss / (b + 1),
                "w2v": total_w2v / (b + 1),
                "rsr": (total_rsr / max(1, (b // rsr_every_n) + 1)) if rsr_weight > 0.0 else 0.0
            })

    return {
        "loss": total_loss / batches_per_epoch,
        "w2v": total_w2v / batches_per_epoch,
        "rsr": total_rsr / max(1, (batches_per_epoch // rsr_every_n)) if rsr_weight > 0.0 else 0.0,
        "things_w2v": total_thw2v / batches_per_epoch if things_w2v_weight > 0.0 else 0.0,
    }

In [10]:
#------------------------------------------------------------------------------
# STEP 3: Create a single shared random initialization (FAIR comparison)
#------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3: Create shared init state for fair Vanilla vs RSR comparison")
print("=" * 70)

init_model = SkipGramWord2Vec(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
init_state = deepcopy(init_model.state_dict())
del init_model
print("[info] Shared init_state captured.")


STEP 3: Create shared init state for fair Vanilla vs RSR comparison
[info] Shared init_state captured.


In [11]:
#------------------------------------------------------------------------------
# STEP 4: Train Vanilla model (Wikipedia only)
#------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4: Train Vanilla Word2Vec (Wikipedia only)")
print("=" * 70)

vanilla_model = SkipGramWord2Vec(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
vanilla_model.load_state_dict(deepcopy(init_state))

vanilla_opt = optim.Adam(vanilla_model.parameters(), lr=LR)

for ep in range(1, EPOCHS + 1):
    stats = train_one_epoch_streaming(
        model=vanilla_model,
        optimizer=vanilla_opt,
        sentence_stream_fn=sentence_stream_factory,
        batches_per_epoch=BATCHES_PER_EPOCH,
        batch_size=BATCH_SIZE,
        window_size=WINDOW_SIZE,
        neg_samples=NEG_SAMPLES,
        neg_dist=NEG_DIST,
        rsr_weight=0.0,                 # <-- vanilla: no RSR
        things_w2v_weight=0.0,
    )
    print(f"[vanilla] epoch {ep}/{EPOCHS} | loss={stats['loss']:.4f} | w2v={stats['w2v']:.4f}")

vanilla_path = MODELS_DIR / "vanilla_w2v.pt"
torch.save(
    {
        "state_dict": vanilla_model.state_dict(),
        "word2idx": word2idx,
        "idx2word": idx2word,
        "embedding_dim": EMBEDDING_DIM,
        "config": {
            "MIN_COUNT": MIN_COUNT,
            "WINDOW_SIZE": WINDOW_SIZE,
            "NEG_SAMPLES": NEG_SAMPLES,
            "SUBSAMPLE_T": SUBSAMPLE_T,
        },
    },
    vanilla_path,
)
print(f"[saved] {vanilla_path}")


STEP 4: Train Vanilla Word2Vec (Wikipedia only)


[vanilla] epoch 1/2 | loss=1.1788 | w2v=1.1788


[vanilla] epoch 2/2 | loss=1.0543 | w2v=1.0543
[saved] models\vanilla_w2v.pt


In [12]:
#------------------------------------------------------------------------------
# STEP 5: Train RSR model (Wikipedia + THINGS RSR loss)  [WITH RAMP-UP]
#------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5: Train RSR Word2Vec (Wikipedia + THINGS-informed loss)")
print("=" * 70)

# -------------------------
# RSR ramp-up constants
# -------------------------
RSR_WARMUP_FRAC = 0.6        # first 60% of total steps: RSR off
RSR_RAMP_FRAC   = 0.2        # next 20%: ramp up to target
RSR_RAMP_CHUNKS = 10         # number of chunks in the ramp (more = smoother)

assert 0.0 <= RSR_WARMUP_FRAC < 1.0
assert 0.0 <= RSR_RAMP_FRAC < 1.0
assert (RSR_WARMUP_FRAC + RSR_RAMP_FRAC) <= 1.0

# -------------------------
# Model + optimizer
# -------------------------
rsr_model = SkipGramWord2Vec(VOCAB_SIZE, EMBEDDING_DIM).to(DEVICE)
rsr_model.load_state_dict(deepcopy(init_state))  # <-- same init as vanilla

rsr_opt = optim.Adam(rsr_model.parameters(), lr=LR)

# -------------------------
# IMPORTANT: keep ONE streaming iterator across all phases
# so we don't restart Wikipedia for each sub-call
# -------------------------
sentence_iter = sentence_stream_factory()

def _same_stream_forever():
    # train_one_epoch_streaming calls sentence_stream_fn() internally;
    # returning the same iterator means we continue where we left off.
    return sentence_iter

# -------------------------
# Helper to accumulate stats across sub-runs
# -------------------------
def _accum_init():
    return {"loss": 0.0, "w2v": 0.0, "rsr": 0.0, "batches": 0}

def _accum_add(acc, stats, batches):
    # stats are means per batch (as returned by your train_one_epoch_streaming)
    acc["loss"] += stats["loss"] * batches
    acc["w2v"]  += stats["w2v"]  * batches
    acc["rsr"]  += stats["rsr"]  * batches
    acc["batches"] += batches

def _accum_finalize(acc):
    if acc["batches"] == 0:
        return {"loss": 0.0, "w2v": 0.0, "rsr": 0.0}
    return {
        "loss": acc["loss"] / acc["batches"],
        "w2v":  acc["w2v"]  / acc["batches"],
        "rsr":  acc["rsr"]  / acc["batches"],
    }

# -------------------------
# Build phase plan for each epoch
# -------------------------
warmup_batches = int(BATCHES_PER_EPOCH * RSR_WARMUP_FRAC)
ramp_batches_total = int(BATCHES_PER_EPOCH * RSR_RAMP_FRAC)
steady_batches = BATCHES_PER_EPOCH - warmup_batches - ramp_batches_total

# Split ramp into chunks (roughly equal size)
if ramp_batches_total > 0:
    base = ramp_batches_total // RSR_RAMP_CHUNKS
    rem  = ramp_batches_total %  RSR_RAMP_CHUNKS
    ramp_chunk_sizes = [base + (1 if i < rem else 0) for i in range(RSR_RAMP_CHUNKS)]
    ramp_chunk_sizes = [b for b in ramp_chunk_sizes if b > 0]
else:
    ramp_chunk_sizes = []

print("[ramp plan]")
print(f"  warmup_batches={warmup_batches} (rsr_weight=0)")
print(f"  ramp_batches_total={ramp_batches_total} in {len(ramp_chunk_sizes)} chunks (0 -> {RSR_WEIGHT})")
print(f"  steady_batches={steady_batches} (rsr_weight={RSR_WEIGHT})")

# -------------------------
# Training loop (epoch = warmup + ramp + steady)
# -------------------------
for ep in range(1, EPOCHS + 1):
    acc = _accum_init()

    # ---- warmup: rsr_weight = 0
    if warmup_batches > 0:
        stats = train_one_epoch_streaming(
            model=rsr_model,
            optimizer=rsr_opt,
            sentence_stream_fn=_same_stream_forever,
            batches_per_epoch=warmup_batches,
            batch_size=BATCH_SIZE,
            window_size=WINDOW_SIZE,
            neg_samples=NEG_SAMPLES,
            neg_dist=NEG_DIST,
            rsr_weight=0.0,
            rsr_every_n=RSR_EVERY_N_BATCHES,
            rsr_pairs_per_step=RSR_PAIRS_PER_STEP,
            soft_rank_strength=SOFT_RANK_STRENGTH,
            things_w2v_weight=THINGS_W2V_WEIGHT,
        )
        _accum_add(acc, stats, warmup_batches)

    # ---- ramp: linearly increase rsr_weight across chunks
    if len(ramp_chunk_sizes) > 0:
        for i, chunk_batches in enumerate(ramp_chunk_sizes, start=1):
            # weight goes from small -> target (avoid exactly 0 by starting at i/num_chunks)
            t = i / float(len(ramp_chunk_sizes))
            w = RSR_WEIGHT * t

            stats = train_one_epoch_streaming(
                model=rsr_model,
                optimizer=rsr_opt,
                sentence_stream_fn=_same_stream_forever,
                batches_per_epoch=chunk_batches,
                batch_size=BATCH_SIZE,
                window_size=WINDOW_SIZE,
                neg_samples=NEG_SAMPLES,
                neg_dist=NEG_DIST,
                rsr_weight=w,
                rsr_every_n=RSR_EVERY_N_BATCHES,
                rsr_pairs_per_step=RSR_PAIRS_PER_STEP,
                soft_rank_strength=SOFT_RANK_STRENGTH,
                things_w2v_weight=THINGS_W2V_WEIGHT,
            )
            _accum_add(acc, stats, chunk_batches)

    # ---- steady: rsr_weight = target
    if steady_batches > 0:
        stats = train_one_epoch_streaming(
            model=rsr_model,
            optimizer=rsr_opt,
            sentence_stream_fn=_same_stream_forever,
            batches_per_epoch=steady_batches,
            batch_size=BATCH_SIZE,
            window_size=WINDOW_SIZE,
            neg_samples=NEG_SAMPLES,
            neg_dist=NEG_DIST,
            rsr_weight=RSR_WEIGHT,
            rsr_every_n=RSR_EVERY_N_BATCHES,
            rsr_pairs_per_step=RSR_PAIRS_PER_STEP,
            soft_rank_strength=SOFT_RANK_STRENGTH,
            things_w2v_weight=THINGS_W2V_WEIGHT,
        )
        _accum_add(acc, stats, steady_batches)

    epoch_stats = _accum_finalize(acc)
    print(
        f"[rsr+ramp] epoch {ep}/{EPOCHS} | loss={epoch_stats['loss']:.4f} | "
        f"w2v={epoch_stats['w2v']:.4f} | rsr={epoch_stats['rsr']:.4f} | "
        f"warmup={RSR_WARMUP_FRAC:.2f} ramp={RSR_RAMP_FRAC:.2f} target={RSR_WEIGHT}"
    )

# -------------------------
# Save model + config
# -------------------------
rsr_path = MODELS_DIR / "rsr_w2v.pt"
torch.save(
    {
        "state_dict": rsr_model.state_dict(),
        "word2idx": word2idx,
        "idx2word": idx2word,
        "embedding_dim": EMBEDDING_DIM,
        "config": {
            "MIN_COUNT": MIN_COUNT,
            "WINDOW_SIZE": WINDOW_SIZE,
            "NEG_SAMPLES": NEG_SAMPLES,
            "SUBSAMPLE_T": SUBSAMPLE_T,
            "RSR_WEIGHT": RSR_WEIGHT,
            "RSR_EVERY_N_BATCHES": RSR_EVERY_N_BATCHES,
            "RSR_PAIRS_PER_STEP": RSR_PAIRS_PER_STEP,
            "SOFT_RANK_STRENGTH": SOFT_RANK_STRENGTH,
            "THINGS_W2V_WEIGHT": THINGS_W2V_WEIGHT,
            "RSR_WARMUP_FRAC": RSR_WARMUP_FRAC,
            "RSR_RAMP_FRAC": RSR_RAMP_FRAC,
            "RSR_RAMP_CHUNKS": RSR_RAMP_CHUNKS,
        },
    },
    rsr_path,
)
print(f"[saved] {rsr_path}")



STEP 5: Train RSR Word2Vec (Wikipedia + THINGS-informed loss)
[ramp plan]
  warmup_batches=6000 (rsr_weight=0)
  ramp_batches_total=2000 in 10 chunks (0 -> 0.01)
  steady_batches=2000 (rsr_weight=0.01)


[rsr+ramp] epoch 1/2 | loss=1.1791 | w2v=1.1790 | rsr=0.1269 | warmup=0.60 ramp=0.20 target=0.01


[rsr+ramp] epoch 2/2 | loss=1.1224 | w2v=1.1223 | rsr=0.0883 | warmup=0.60 ramp=0.20 target=0.01
[saved] models\rsr_w2v.pt


In [13]:
#------------------------------------------------------------------------------
# STEP 6: SimLex-999 evaluation (Spearman correlation)
#------------------------------------------------------------------------------

def load_simlex(path: Path):
    """
    Supports common SimLex formats:
      - TSV with header: word1 word2 SimLex999
      - CSV with 'word1','word2','SimLex999' columns
    """
    # Try pandas auto-detect
    df = pd.read_csv(path, sep=None, engine="python")
    cols = [c.lower() for c in df.columns]

    # Heuristic column picks
    if "word1" in cols and "word2" in cols:
        w1_col = df.columns[cols.index("word1")]
        w2_col = df.columns[cols.index("word2")]
    else:
        # fallback: first two columns
        w1_col, w2_col = df.columns[:2]

    # score column
    score_col = None
    for candidate in ["simlex999", "simlex", "score", "similarity"]:
        if candidate in cols:
            score_col = df.columns[cols.index(candidate)]
            break
    if score_col is None:
        score_col = df.columns[2]  # common layout

    df = df[[w1_col, w2_col, score_col]].copy()
    df.columns = ["word1", "word2", "score"]
    df["word1"] = df["word1"].astype(str).str.lower()
    df["word2"] = df["word2"].astype(str).str.lower()
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df = df.dropna(subset=["score"])
    return df

@torch.no_grad()
def simlex_spearman(model: SkipGramWord2Vec, simlex_df: pd.DataFrame, restrict_to_things: bool = False):
    """
    Compute Spearman between model cosine similarities and SimLex human scores.
    If restrict_to_things=True, only include pairs where both words are THINGS concepts in vocab.
    """
    model.eval()

    # Get in-embedding matrix
    W = model.in_embed.weight.detach()  # (V, D)

    # Normalize for cosine
    Wn = W / (W.norm(dim=1, keepdim=True) + 1e-8)

    things_set = set(valid_concepts)

    sims = []
    scores = []
    covered = 0

    for _, row in simlex_df.iterrows():
        w1 = row["word1"]
        w2 = row["word2"]
        if restrict_to_things and (w1 not in things_set or w2 not in things_set):
            continue

        i = word2idx.get(w1, 0)
        j = word2idx.get(w2, 0)
        if i == 0 or j == 0:
            continue

        s = float((Wn[i] * Wn[j]).sum().item())
        sims.append(s)
        scores.append(float(row["score"]))
        covered += 1

    if covered < 10:
        return {
            "rho": np.nan,
            "p": np.nan,
            "n": covered,
        }

    rho, p = spearmanr(sims, scores)
    return {
        "rho": float(rho),
        "p": float(p),
        "n": int(covered),
    }

print("\n" + "=" * 70)
print("STEP 6: SimLex-999 evaluation")
print("=" * 70)

if not SIMLEX_PATH.exists():
    print(f"[warn] SimLex file not found at: {SIMLEX_PATH}")
    print("       Update SIMLEX_PATH at the top of the notebook/script.")
else:
    simlex_df = load_simlex(SIMLEX_PATH)
    print(f"[info] Loaded SimLex rows: {len(simlex_df):,}")

    v_all = simlex_spearman(vanilla_model, simlex_df, restrict_to_things=False)
    r_all = simlex_spearman(rsr_model, simlex_df, restrict_to_things=False)

    v_th = simlex_spearman(vanilla_model, simlex_df, restrict_to_things=True)
    r_th = simlex_spearman(rsr_model, simlex_df, restrict_to_things=True)

    print("\n[SimLex-999 | all covered pairs]")
    print(f"  vanilla: rho={v_all['rho']:.4f} (n={v_all['n']})")
    print(f"  rsr:     rho={r_all['rho']:.4f} (n={r_all['n']})")

    print("\n[SimLex-999 | THINGS-only pairs]")
    print(f"  vanilla: rho={v_th['rho']:.4f} (n={v_th['n']})")
    print(f"  rsr:     rho={r_th['rho']:.4f} (n={r_th['n']})")


STEP 6: SimLex-999 evaluation
[info] Loaded SimLex rows: 999

[SimLex-999 | all covered pairs]
  vanilla: rho=0.2588 (n=999)
  rsr:     rho=0.2675 (n=999)

[SimLex-999 | THINGS-only pairs]
  vanilla: rho=0.0606 (n=104)
  rsr:     rho=0.1328 (n=104)


In [14]:
#------------------------------------------------------------------------------
# STEP 7 (optional): qualitative nearest neighbors (SimLex vocab only)
#------------------------------------------------------------------------------

import pandas as pd

# --- build SimLex vocab once ---
def load_simlex_vocab(simlex_path):
    df = pd.read_csv(simlex_path, sep=None, engine="python")
    cols = [c.lower() for c in df.columns]

    if "word1" in cols and "word2" in cols:
        w1c = df.columns[cols.index("word1")]
        w2c = df.columns[cols.index("word2")]
    else:
        w1c, w2c = df.columns[:2]

    vocab = set(df[w1c].astype(str).str.lower()) | set(df[w2c].astype(str).str.lower())
    return sorted(vocab)

simlex_vocab = load_simlex_vocab(SIMLEX_PATH)

# map to indices (drop OOV)
SIMLEX_ALLOWED_IDXS = torch.tensor(
    sorted({word2idx[w] for w in simlex_vocab if w in word2idx}),
    dtype=torch.long,
    device=DEVICE
)

print(f"[info] SimLex vocab size (in model vocab): {len(SIMLEX_ALLOWED_IDXS)}")

# --- restricted NN function ---
@torch.no_grad()
def nearest_neighbors_simlex_only(model: SkipGramWord2Vec, query_word: str, topk: int = 10):
    model.eval()
    query_word = query_word.lower()
    qi = word2idx.get(query_word, 0)
    if qi == 0:
        return None

    W = model.in_embed.weight.detach()
    Wn = W / (W.norm(dim=1, keepdim=True) + 1e-8)
    qv = Wn[qi]  # (D,)

    # similarities restricted to SimLex vocab
    sims = torch.matmul(Wn[SIMLEX_ALLOWED_IDXS], qv)

    # exclude self if present
    mask_self = SIMLEX_ALLOWED_IDXS == qi
    if mask_self.any():
        sims = sims.clone()
        sims[mask_self] = -1e9

    vals, rel_idxs = torch.topk(sims, k=min(topk, sims.shape[0]))
    nn_idxs = SIMLEX_ALLOWED_IDXS[rel_idxs]

    return [(idx2word[int(i)], float(v)) for i, v in zip(nn_idxs.cpu(), vals.cpu())]

# --- run qualitative comparison ---
test_words = ["cat", "dog", "king", "queen", "car", "bicycle"]

for w in test_words:
    nn_v = nearest_neighbors_simlex_only(vanilla_model, w, topk=8)
    nn_r = nearest_neighbors_simlex_only(rsr_model, w, topk=8)

    if nn_v is None or nn_r is None:
        print(f"\n[w={w}] not in vocab")
        continue

    print(f"\n[w={w}] vanilla (SimLex-only):", nn_v)
    print(f"[w={w}] rsr (SimLex-only):    ", nn_r)


[info] SimLex vocab size (in model vocab): 1028

[w=cat] vanilla (SimLex-only): [('dog', 0.4260363280773163), ('goat', 0.37050294876098633), ('mouse', 0.34179866313934326), ('bird', 0.3244895040988922), ('pet', 0.317720502614975), ('rabbit', 0.30717724561691284), ('rat', 0.2980539798736572), ('steak', 0.28829333186149597)]
[w=cat] rsr (SimLex-only):     [('dog', 0.41921085119247437), ('rabbit', 0.4014335870742798), ('ant', 0.38939082622528076), ('pet', 0.3844807744026184), ('goat', 0.3387238681316376), ('bird', 0.2973605692386627), ('bee', 0.29664501547813416), ('hawk', 0.2924734055995941)]

[w=dog] vanilla (SimLex-only): [('cat', 0.4260363280773163), ('goat', 0.40382856130599976), ('animal', 0.3995691239833832), ('horse', 0.39505335688591003), ('girl', 0.3588644564151764), ('hound', 0.35654571652412415), ('pet', 0.35530713200569153), ('rabbit', 0.34951549768447876)]
[w=dog] rsr (SimLex-only):     [('goat', 0.47421926259994507), ('rabbit', 0.4544466733932495), ('hound', 0.4467997252941

In [15]:
#------------------------------------------------------------------------------
# STEP 6A: Benchmark paths (MATCHING YOUR PROJECT TREE)
#------------------------------------------------------------------------------

from pathlib import Path

ROOT_DIR = Path(".")
DATA_DIR = ROOT_DIR / "data"

# SimLex-999 is in its own folder at project root (per your screenshot)
SIMLEX_PATH = ROOT_DIR / "SimLex-999" / "SimLex-999.txt"

# SimVerb-3500 file (canonical)
SIMVERB_PATH = DATA_DIR / "simverb-3500-data" / "data" / "SimVerb-3500.txt"

# MEN lemma full (your folder nesting)
MEN_PATH = DATA_DIR / "MEN" / "MEN" / "MEN_dataset_lemma_form_full"

# Google analogies
ANALOGY_PATH = DATA_DIR / "questions-words.txt"

for name, p in [
    ("SimLex", SIMLEX_PATH),
    ("SimVerb", SIMVERB_PATH),
    ("MEN", MEN_PATH),
    ("Analogies", ANALOGY_PATH),
]:
    print(f"{name:10s}: {p}  exists={p.exists()}")


SimLex    : SimLex-999\SimLex-999.txt  exists=True
SimVerb   : data\simverb-3500-data\data\SimVerb-3500.txt  exists=True
MEN       : data\MEN\MEN\MEN_dataset_lemma_form_full  exists=True
Analogies : data\questions-words.txt  exists=True


In [16]:
#------------------------------------------------------------------------------
# STEP 6B: Core helpers (Spearman + similarity evaluator + loaders)
#------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import torch

def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    xr = pd.Series(x).rank(method="average").to_numpy()
    yr = pd.Series(y).rank(method="average").to_numpy()

    xr = xr - xr.mean()
    yr = yr - yr.mean()
    denom = (np.sqrt((xr**2).sum()) * np.sqrt((yr**2).sum()))
    if denom == 0:
        return np.nan
    return float((xr * yr).sum() / denom)

@torch.no_grad()
def eval_similarity_dataset(model, pairs, word2idx, name="dataset"):
    """
    Spearman rho between gold ratings and cosine similarity of in-embeddings.
    Evaluates only covered pairs.
    """
    model.eval()
    W = model.in_embed.weight.detach()
    Wn = W / (W.norm(dim=1, keepdim=True) + 1e-8)

    gold = []
    pred = []
    covered = 0

    for w1, w2, y in pairs:
        i = word2idx.get(w1, 0)
        j = word2idx.get(w2, 0)
        if i == 0 or j == 0:
            continue
        sim = float((Wn[i] * Wn[j]).sum().item())
        gold.append(float(y))
        pred.append(sim)
        covered += 1

    rho = spearman_corr(gold, pred) if covered > 1 else np.nan
    return {"name": name, "rho": rho, "n": covered}

def load_wordpair_dataset_auto(path: Path, dataset_name: str):
    """
    Robust loader for similarity datasets (SimLex / SimVerb etc.).
    Handles:
    - headers (e.g. SimLex)
    - flexible delimiters (tab/space/comma)
    - extra columns
    Chooses the first mostly-numeric column after the first two word columns.
    """
    if not path.exists():
        raise FileNotFoundError(f"{dataset_name} not found at: {path}")

    df = pd.read_csv(path, sep=None, engine="python")

    # If delimiter guess fails and you get one big column, try whitespace split
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=r"\s+", engine="python", header=None)

    if df.shape[1] < 3:
        raise ValueError(f"{dataset_name}: expected >=3 columns, got {df.shape[1]}")

    w1 = df.iloc[:, 0].astype(str).str.lower()
    w2 = df.iloc[:, 1].astype(str).str.lower()

    score_series = None
    chosen_col = None
    for c in range(2, df.shape[1]):
        s = pd.to_numeric(df.iloc[:, c], errors="coerce")
        if float(s.notna().mean()) > 0.90:
            score_series = s
            chosen_col = c
            break

    if score_series is None:
        score_series = pd.to_numeric(df.iloc[:, 2], errors="coerce")
        chosen_col = 2

    mask = score_series.notna()
    out = list(zip(w1[mask].tolist(), w2[mask].tolist(), score_series[mask].astype(float).tolist()))
    if len(out) == 0:
        raise ValueError(f"{dataset_name}: no valid rows parsed (score_col={chosen_col})")

    print(f"[info] {dataset_name}: parsed {len(out)} rows (score_col={chosen_col})")
    return out

def load_men_dataset(path: Path):
    """
    MEN is reliably whitespace-separated; first 3 columns are word1 word2 score.
    """
    if not path.exists():
        raise FileNotFoundError(f"MEN file not found: {path}")

    df = pd.read_csv(path, sep=r"\s+", engine="python", header=None)
    if df.shape[1] < 3:
        raise ValueError(f"MEN: expected >=3 columns, got {df.shape[1]}")

    w1 = df.iloc[:, 0].astype(str).str.lower()
    w2 = df.iloc[:, 1].astype(str).str.lower()
    scores = pd.to_numeric(df.iloc[:, 2], errors="coerce")

    mask = scores.notna()
    pairs = list(zip(
        w1[mask].tolist(),
        w2[mask].tolist(),
        scores[mask].astype(float).tolist()
    ))

    print(f"[info] MEN: parsed {len(pairs)} rows")
    return pairs


In [17]:
#------------------------------------------------------------------------------
# STEP 6C: Load datasets (SimLex / SimVerb / MEN)
#------------------------------------------------------------------------------

simlex_pairs  = load_wordpair_dataset_auto(SIMLEX_PATH,  "SimLex-999")
simverb_pairs = load_wordpair_dataset_auto(SIMVERB_PATH, "SimVerb-3500")
men_pairs     = load_men_dataset(MEN_PATH)

print("[info] Loaded rows:",
      f"SimLex={len(simlex_pairs)}",
      f"SimVerb={len(simverb_pairs)}",
      f"MEN={len(men_pairs)}")


[info] SimLex-999: parsed 999 rows (score_col=3)
[info] SimVerb-3500: parsed 3499 rows (score_col=3)
[info] MEN: parsed 3000 rows
[info] Loaded rows: SimLex=999 SimVerb=3499 MEN=3000


In [18]:
#------------------------------------------------------------------------------
# STEP 6D: SimLex-999 evaluation (explicit print cell)
#------------------------------------------------------------------------------

v_simlex = eval_similarity_dataset(vanilla_model, simlex_pairs, word2idx, name="SimLex-999")
r_simlex = eval_similarity_dataset(rsr_model,     simlex_pairs, word2idx, name="SimLex-999")

print("\n[SimLex-999 | all covered pairs]")
print(f"  vanilla: rho={v_simlex['rho']:.4f} (n={v_simlex['n']})")
print(f"  rsr:     rho={r_simlex['rho']:.4f} (n={r_simlex['n']})")



[SimLex-999 | all covered pairs]
  vanilla: rho=0.2588 (n=999)
  rsr:     rho=0.2675 (n=999)


In [19]:
#------------------------------------------------------------------------------
# STEP 6E: SimVerb-3500 evaluation (explicit print cell)
#------------------------------------------------------------------------------

v_simverb = eval_similarity_dataset(vanilla_model, simverb_pairs, word2idx, name="SimVerb-3500")
r_simverb = eval_similarity_dataset(rsr_model,     simverb_pairs, word2idx, name="SimVerb-3500")

print("\n[SimVerb-3500 | all covered pairs]")
print(f"  vanilla: rho={v_simverb['rho']:.4f} (n={v_simverb['n']})")
print(f"  rsr:     rho={r_simverb['rho']:.4f} (n={r_simverb['n']})")



[SimVerb-3500 | all covered pairs]
  vanilla: rho=0.1539 (n=3494)
  rsr:     rho=0.1457 (n=3494)


In [20]:
#------------------------------------------------------------------------------
# STEP 6F: MEN evaluation (explicit print cell)
#------------------------------------------------------------------------------

v_men = eval_similarity_dataset(vanilla_model, men_pairs, word2idx, name="MEN (lemma full)")
r_men = eval_similarity_dataset(rsr_model,     men_pairs, word2idx, name="MEN (lemma full)")

print("\n[MEN (lemma full) | all covered pairs]")
print(f"  vanilla: rho={v_men['rho']:.4f} (n={v_men['n']})")
print(f"  rsr:     rho={r_men['rho']:.4f} (n={r_men['n']})")



[MEN (lemma full) | all covered pairs]
  vanilla: rho=nan (n=0)
  rsr:     rho=nan (n=0)


In [21]:
#------------------------------------------------------------------------------
# STEP 6G: Google analogies eval (questions-words.txt) using 3CosAdd
#------------------------------------------------------------------------------

@torch.no_grad()
def eval_analogies_3cosadd(
    model,
    analogy_path: Path,
    word2idx: dict,
    device=DEVICE,
    topk=1,
    restrict_sections=None,   # e.g. {"capital-common-countries", "family"}
    max_questions=None,
):
    if not analogy_path.exists():
        raise FileNotFoundError(f"Analogy file not found: {analogy_path}")

    model.eval()
    W = model.in_embed.weight.detach().to(device)
    Wn = W / (W.norm(dim=1, keepdim=True) + 1e-8)

    def best_pred(a_i, b_i, c_i, banned):
        q = Wn[b_i] - Wn[a_i] + Wn[c_i]
        q = q / (q.norm() + 1e-8)
        sims = torch.mv(Wn, q)
        for bi in banned:
            sims[bi] = -1e9
        _, idxs = torch.topk(sims, k=topk)
        return idxs.tolist()

    section = None
    total_eval = 0
    total_correct = 0
    per_section = {}

    with analogy_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(":"):
                section = line[1:].strip()
                per_section.setdefault(section, {"eval": 0, "correct": 0, "oov": 0})
                continue

            if section is None:
                continue
            if restrict_sections is not None and section not in restrict_sections:
                continue

            a, b, c, d = line.lower().split()
            a_i = word2idx.get(a, 0)
            b_i = word2idx.get(b, 0)
            c_i = word2idx.get(c, 0)
            d_i = word2idx.get(d, 0)

            if 0 in (a_i, b_i, c_i, d_i):
                per_section[section]["oov"] += 1
                continue

            preds = best_pred(a_i, b_i, c_i, banned=[0, a_i, b_i, c_i])
            per_section[section]["eval"] += 1
            total_eval += 1

            if d_i in preds:
                per_section[section]["correct"] += 1
                total_correct += 1

            if max_questions is not None and total_eval >= max_questions:
                break

    acc = (total_correct / total_eval) if total_eval > 0 else float("nan")
    return {"overall": {"acc": acc, "eval": total_eval, "correct": total_correct},
            "per_section": per_section}

def print_analogy_results(tag, res, top_sections=8):
    print(f"\n[{tag}] overall acc@1={res['overall']['acc']:.4f} (correct={res['overall']['correct']}/{res['overall']['eval']})")
    secs = list(res["per_section"].items())
    secs.sort(key=lambda x: x[1]["eval"], reverse=True)
    print("  top sections by #eval:")
    for sec, st in secs[:top_sections]:
        acc = (st["correct"] / st["eval"]) if st["eval"] > 0 else float("nan")
        print(f"   - {sec:30s} acc={acc:.4f}  eval={st['eval']}  oov={st['oov']}")

v_ana = eval_analogies_3cosadd(vanilla_model, ANALOGY_PATH, word2idx, topk=1)
r_ana = eval_analogies_3cosadd(rsr_model,     ANALOGY_PATH, word2idx, topk=1)

print_analogy_results("vanilla", v_ana)
print_analogy_results("rsr", r_ana)



[vanilla] overall acc@1=0.3375 (correct=6596/19544)
  top sections by #eval:
   - capital-world                  acc=0.5882  eval=4524  oov=0
   - city-in-state                  acc=0.3421  eval=2467  oov=0
   - gram6-nationality-adjective    acc=0.8193  eval=1599  oov=0
   - gram7-past-tense               acc=0.2301  eval=1560  oov=0
   - gram3-comparative              acc=0.1051  eval=1332  oov=0
   - gram8-plural                   acc=0.2252  eval=1332  oov=0
   - gram4-superlative              acc=0.0303  eval=1122  oov=0
   - gram5-present-participle       acc=0.1695  eval=1056  oov=0

[rsr] overall acc@1=0.3164 (correct=6183/19544)
  top sections by #eval:
   - capital-world                  acc=0.5411  eval=4524  oov=0
   - city-in-state                  acc=0.3089  eval=2467  oov=0
   - gram6-nationality-adjective    acc=0.7736  eval=1599  oov=0
   - gram7-past-tense               acc=0.2301  eval=1560  oov=0
   - gram3-comparative              acc=0.1186  eval=1332  oov=0
   

In [22]:
#------------------------------------------------------------------------------
# STEP 6H: Final scoreboard (collate all metrics into one table)
#------------------------------------------------------------------------------

def build_scoreboard():
    rows = []

    rows.append({
        "benchmark": "SimLex-999",
        "metric": "spearman_rho",
        "vanilla": v_simlex["rho"],
        "rsr": r_simlex["rho"],
        "delta(rsr-vanilla)": r_simlex["rho"] - v_simlex["rho"],
        "n_vanilla": v_simlex["n"],
        "n_rsr": r_simlex["n"],
    })

    rows.append({
        "benchmark": "SimVerb-3500",
        "metric": "spearman_rho",
        "vanilla": v_simverb["rho"],
        "rsr": r_simverb["rho"],
        "delta(rsr-vanilla)": r_simverb["rho"] - v_simverb["rho"],
        "n_vanilla": v_simverb["n"],
        "n_rsr": r_simverb["n"],
    })

    rows.append({
        "benchmark": "MEN (lemma full)",
        "metric": "spearman_rho",
        "vanilla": v_men["rho"],
        "rsr": r_men["rho"],
        "delta(rsr-vanilla)": r_men["rho"] - v_men["rho"],
        "n_vanilla": v_men["n"],
        "n_rsr": r_men["n"],
    })

    rows.append({
        "benchmark": "Google analogies",
        "metric": "accuracy@1",
        "vanilla": v_ana["overall"]["acc"],
        "rsr": r_ana["overall"]["acc"],
        "delta(rsr-vanilla)": r_ana["overall"]["acc"] - v_ana["overall"]["acc"],
        "n_vanilla": v_ana["overall"]["eval"],
        "n_rsr": r_ana["overall"]["eval"],
    })

    return pd.DataFrame(rows)

scoreboard = build_scoreboard()
scoreboard


,benchmark,metric,vanilla,rsr,delta(rsr-vanilla),n_vanilla,n_rsr
0,SimLex-999,spearman_rho,0.258840,0.267526,0.008686,999,999
1,SimVerb-3500,spearman_rho,0.153872,0.145678,-0.008194,3494,3494
2,MEN (lemma full),spearman_rho,NaN,NaN,NaN,0,0
3,Google analogies,accuracy@1,0.337495,0.316363,-0.021132,19544,19544
